# 6. After the milestone: tuning and final evaluation

Use only after the baseline is understood. This optional run compares LR, RBF SVM and RF settings using a small inner three-fold subject-wise search, repeated inside outer LOSO. Standardization is inside the searched pipeline.

The feature set is deliberately fixed to the milestone time features and 2-second windows. Notebook 5's globally selected feature winner must not be substituted and described as unbiased nested evaluation: that would require feature selection inside the inner search too.

RBF SVM can be slow on a CPU. Training is disabled by default.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), Path.cwd() / 'harth_project', Path.cwd().parent]
            if (p / 'har_utils.py').exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from har_utils import *

## Define the nested selection procedure

This function is used only in notebook 06, so its implementation lives here. The outer fold holds out a person. Inner folds select settings using only the other people. Standardization stays inside the searched pipeline.

Read the candidate grid, subject assertions, refit and prediction steps, then the per-fold checkpoints. Defining the function does not run the search.

In [ ]:
import json
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.model_selection import GridSearchCV, GroupKFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def run_nested_tuning(X, meta):
    """Select model family and parameters inside each outer training split.

    The feature representation and window length must be fixed before this run.
    This does not validate feature choices already selected on all participants.
    """
    pipeline = Pipeline([('scale', StandardScaler()),
                         ('model', LogisticRegression(max_iter=1500))])
    # Small search by design. RBF SVM can still be slow on the full dataset.
    candidates = [
        {'model': [LogisticRegression(max_iter=1500, random_state=SEED)],
         'model__C': [.1, 1]},
        {'model': [SVC(kernel='rbf', cache_size=512)],
         'model__C': [1, 10], 'model__gamma': ['scale']},
        {'model': [RandomForestClassifier(n_estimators=100,
                    min_samples_leaf=2, n_jobs=4, random_state=SEED)],
         'model__max_depth': [12, 20],
         'model__class_weight': [None, 'balanced_subsample']},
    ]
    scorer = make_scorer(f1_score, labels=LABELS, average='macro', zero_division=0)
    y = meta.label.to_numpy()
    groups = meta.subject.to_numpy()
    rows, predictions, searches = [], [], []
    out = ROOT / 'results'
    for fold, (train, test) in enumerate(LeaveOneGroupOut().split(X, y, groups)):
        assert not set(groups[train]) & set(groups[test])
        inner = list(GroupKFold(3).split(X.iloc[train], y[train], groups[train]))
        for inner_train, inner_test in inner:
            assert not set(groups[train][inner_train]) & set(groups[train][inner_test])
        search = GridSearchCV(pipeline, candidates, scoring=scorer,
                              cv=inner, n_jobs=1, refit=True, error_score='raise')
        start = time.perf_counter()
        search.fit(X.iloc[train], y[train])
        predicted = search.predict(X.iloc[test])
        rows.append({'experiment': 'Nested_model_search', 'protocol': 'loso',
            'fold': fold, 'test_subjects': ','.join(sorted(set(groups[test]))),
            'train_windows': len(train), 'test_windows': len(test),
            'accuracy': accuracy_score(y[test], predicted),
            'macro_f1': f1_score(y[test], predicted, labels=LABELS,
                                 average='macro', zero_division=0),
            'present_class_macro_f1': f1_score(y[test], predicted,
                labels=sorted(set(y[test])), average='macro', zero_division=0),
            'seconds': time.perf_counter() - start})
        prediction = meta.iloc[test].copy()
        prediction['window_id'] = test
        prediction['predicted'] = predicted
        prediction['fold'] = fold
        predictions.append(prediction)
        searches.append({'fold': fold, 'held_out': sorted(set(groups[test])),
                         'best_parameters': search.best_params_,
                         'inner_macro_f1': search.best_score_})
        pd.DataFrame(search.cv_results_).to_csv(
            out / f'Nested_model_search_inner_fold_{fold}.csv', index=False)
        # Checkpoint each completed outer fold; incomplete runs are explicit.
        pd.DataFrame(rows).to_csv(out / 'Nested_model_search_loso_folds.csv', index=False)
        pd.concat(predictions, ignore_index=True).to_csv(
            out / 'Nested_model_search_loso_predictions.csv', index=False)
        (out / 'Nested_model_search_config.json').write_text(json.dumps({
            'seed': SEED, 'feature_columns': list(X.columns),
            'candidate_grid': candidates, 'completed_outer_folds': len(rows),
            'expected_outer_folds': len(set(groups)), 'selections': searches,
        }, default=str, indent=2))
        print(f'Completed nested fold {fold + 1}/{len(set(groups))}', flush=True)
    return pd.DataFrame(rows)

In [ ]:
RUN_TUNING = False
if RUN_TUNING:
    X, meta = load_windows(2)
    tuned_scores = run_nested_tuning(X[feature_columns(X)], meta)
else:
    print('Nested tuning is deferred. No training started.')

## Compare against the fixed RF baseline

Compare subject-level scores on matching outer folds. Nested scores evaluate the model-selection procedure; different outer folds may choose different model families. These are the same 22 participants used for milestone exploration, not an untouched external test cohort.

In [ ]:
path = ROOT / 'results/Nested_model_search_loso_folds.csv'
if path.exists():
    tuned = pd.read_csv(path)
    if tuned.test_subjects.nunique() != 22:
        print('Incomplete nested run: do not report this as full LOSO.')
    else:
        baseline = pd.read_csv(ROOT / 'results/RF_loso_folds.csv')
        display(summarize(pd.concat([baseline, tuned])).round(4))
        paired = baseline[['test_subjects','macro_f1']].merge(
            tuned[['test_subjects','macro_f1']], on='test_subjects',
            suffixes=('_baseline','_tuned'), validate='one_to_one')
        paired['difference'] = paired.macro_f1_tuned - paired.macro_f1_baseline
        display(paired.sort_values('difference').round(4))
        print('Mean paired F1 change:', paired.difference.mean())
else:
    print('No tuning results yet.')

## Final checks when results exist

Inspect rare-class recall, between-subject variation and the saved per-fold selections. Report the candidate grid and both inner and outer splitting rules. Do not change the grid after inspecting outer scores and continue calling the scores an unbiased selection estimate. A CNN remains optional.